# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VikasKajeepeta2005/FlyRank-Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

I am working with the **Refresh / Content Opportunity Scoring** lane.

**Unit of analysis:** One row represents one content item/page for a given observation period.

**Time window:** I will use a mid-panel month, **March 2026 (`month=2026-03`)**, for development and verification.

The goal is to use information available during this observation period to rank pages for content review. I will avoid using the final month as a development window because it can overlap with a future outcome.

## 2. Fields: feature / label / context / excluded

### Features

I plan to use observable search and content signals such as:

- `impressions`
- `clicks`
- `sessions`
- `avg_position`
- `content_age_days`

These are useful because they describe page visibility, traffic, search position, and content age.

### Label / Proxy

For the starter version, the decline indicator can be used as a proxy for identifying pages showing negative movement.

### Context

Fields such as content type, intent, age tier, and freshness tier can provide context for interpreting page performance.

### Excluded

I will exclude product decision outputs such as `health_score`, `priority_score`, and `action_type`. These represent existing decisions rather than independent observable signals and could create circular results or leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    """
).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



In [6]:
# Query 1 — Verify the data grain

query1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_daily_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

result1 = con.sql(query1)
result1.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────┬───────────────┬───────────────────┐
│ total_rows │ clients │ content_items │ unique_daily_rows │
│   int64    │  int64  │     int64     │       int64       │
├────────────┼─────────┼───────────────┼───────────────────┤
│    9841378 │      55 │        331437 │           9841378 │
└────────────┴─────────┴───────────────┴───────────────────┘



In [7]:
# Query 2 — Verify row count and date window

query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

result2 = con.sql(query2)
result2.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [8]:
# Query 3 — Check data availability

query3 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= '2026-03-01'
  AND report_date < '2026-04-01'
"""

result3 = con.sql(query3)
result3.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┬────────────────────────┬────────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ gsc_not_available_rows │ ga4_not_available_rows │
│   int64    │       int64        │       int64        │         int64          │         int64          │
├────────────┼────────────────────┼────────────────────┼────────────────────────┼────────────────────────┤
│    9841378 │            3611061 │             413966 │                6230317 │                9427412 │
└────────────┴────────────────────┴────────────────────┴────────────────────────┴────────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

The March 2026 slice has an unbalanced history across clients and content items.

GSC and GA4 are not available for every row. In particular, the March results show that GA4 is available for fewer rows than GSC.

Missing analytics data should not automatically be treated as zero performance.

The March 2026 slice is also only one month of the warehouse, so the results may not represent every time period or seasonal pattern.

For future ML work, feature and target windows must be separated to avoid data leakage.

In [9]:
# Supporting check for the data limitations

print("March 2026 GSC available rows:", 3611061)
print("March 2026 GA4 available rows:", 413966)

print("GA4 availability is substantially lower than GSC availability.")

March 2026 GSC available rows: 3611061
March 2026 GA4 available rows: 413966
GA4 availability is substantially lower than GSC availability.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.